# **Mapeamento do Ambiente e Modelagem de Dados**
<font size=3>
    
O objetivo desta fase é transformar o mundo físico (o prédio da faculdade) em uma representação matemática e estruturada de dados que o código Kotlin consiga processar. Sem um modelo de dados preciso, não é possível calcular onde a pessoa está nem para onde ela deve ir.

### **1. Sistema de Coordenadas Cartesianas Indoor ($X, Y$):**
<font size=3>

Para navegar dentro de um prédio, não utilizamos latitude e longitude (comuns no GPS externo), pois a precisão métrica necessária é muito maior. Em vez disso, adotamos um **Sistema de Coordenadas Cartesianas Local**:

* **Definição da Origem $(0, 0)$:** Escolhe-se um canto fixo da planta do prédio (geralmente o canto inferior esquerdo do piso) como o ponto $(0,0)$.

* **Unidade de Medida:** A escala deve ser estritamente em **metros**. Se uma sala está a 10,5 metros à direita e
4 metros acima do ponto de origem, suas coordenadas serão $X = 10.5$ e $Y = 4.0$.

* **Piso/Andar ($Z$):** Para prédios com múltiplos andares, adiciona-se uma variável inteira `piso` ($Z = 0$ para o térreo, $Z = 1$ para o 1º andar, etc.).

### **2. Modelagem dos Pontos de Acesso Wi-Fi (*Access Points - APs*):**
<font size=3>

Para realizar a trilateração na Fase 2, o algoritmo precisa saber exatamente onde cada roteador Wi-Fi está fixado no prédio e quais são suas características de transmissão.

* **BSSID (Endereço MAC):** O identificador único e físico do roteador (ex: `"00:1A:2B:3C:4D:5E"`). Não devemos confiar apenas no SSID (nome da rede), pois vários roteadores no mesmo prédio podem compartilhar o mesmo nome (ex: "WiFi-UFRN").

* **Coordenadas de Posicionamento:** A localização exata do roteador no teto ou parede em $(X, Y, Z)$.

* **Parâmetros de Sinal de Referência:**

* $P_0$ (ou $A$): O valor de RSSI esperado a exatamente 1 metro de distância do roteador (geralmente entre $-30 \text{ dBm}$ e $-45 \text{ dBm}$).

* $n$ (Expoente de Perda do Caminho): Constante que representa o nível de atenuação das paredes e obstáculos do ambiente (varia de $2.0$ para espaço livre a $3.0\text{–}4.0$ em prédios com muitas paredes de alvenaria).

In [ ]:
data class AccessPoint(
    val bssid: String,
    val ssid: String,
    val x: Double,
    val y: Double,
    val floor: Int,
    val referenceRssi1m: Int = -40, // Valor padrão em dBm
    val attenuationExponent: Double = 2.5 // Fator do ambiente
)

### **3. Teoria dos Grafos Aplicada à Navegação Indoor:**
<font size=3>

O mapa do prédio não deve ser desenhado como uma imagem flexível, mas sim modelado como um **Grafo Direcionado e Ponderado**.

* **Vértices ou Nós (Nodes):** Representam pontos estratégicos do piso onde a pessoa pode estar, mudar de direção ou parar.
    * *Exemplos:* Cruzamento de corredores, portas de salas, entrada do elevador, início e fim de rampas.
    <br>

* **Arestas (Edges):** Representam os caminhos navegáveis que conectam dois nós.
    * *Pesos (Weights):* A distância física em metros entre os nós, calculada pela distância euclidiana:
    $$d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}$$
    
* **Atributos de Acessibilidade:** Essenciais para guiar pessoas com deficiência. Cada aresta deve conter informações de acessibilidade (ex: `temEscada`, `temRampa`, `larguraCorredorMetros`).

In [ ]:
data class Node(val id: String,
                val name: String,
                val x: Double,
                val y: Double,
                val floor: Int)

data class Edge(val originNodeId: String,
                val destinationNodeId: String,
                val distanceMeters: Double,
                val accessibleVisualImpairment: Boolean = true,
                val hasTactilePaving: Boolean = false)


## **4. Pontos de Interesse (POIs) e Semântica do Espaço:**
<font size=3>

Pessoas com baixa visão se beneficiam imensamente de referências contextuais e auditivas. O modelo precisa mapear **Pontos de Interesse (POIs)** ancorados aos nós do grafo.

* **Tipos de POIs:** Banheiros, catracas, balcões de informação, bebedouros, extintores de incêndio e saídas de emergência.

* **Descrição Semântica:** Texto explicativo que o leitor de voz sintetizada usará para dar dicas ambientais (ex: "A porta da secretaria possui maçaneta redonda e fica à direita do pilar").

In [ ]:
enum class PoiType {CLASSROOM, RESTROOM, WATER_FOUNTAIN, ELEVATOR, STAIRS, ENTRANCE_EXIT, INFO_DESK}

data class PointOfInterest(val id: String,
                           val anchoredNodeId: String, // ID do Nó do grafo onde o POI está localizado
                           val name: String,
                           val type: PoiType,
                           val accessibleDescription: String)


### **5. Agrupamento e Serialização (Estrutura do `map.json`):**
<font size=3>

Toda a infraestrutura do prédio é agrupada em uma classe central chamada `MapaPredio` para ser salva e exportada no formato **JSON**.

In [ ]:
data class BuildingMap(val buildingId: String,
                       val buildingName: String,
                       val accessPoints: List<AccessPoint>,
                       val nodes: List<Node>,
                       val edges: List<Edge>,
                       val pointsOfInterest: List<PointOfInterest>)


#### Exemplo da Estrutura do Arquivo `map.json`:
<font size=2>

```json
{
  "buildingId": "BTI_BLOCO_A",
  "buildingName": "Bloco A - Bacharelado em TI",
  "accessPoints": [
    {
      "bssid": "AA:BB:CC:11:22:33",
      "ssid": "WIFI_UFRN",
      "x": 0.0,
      "y": 12.5,
      "floor": 0,
      "referenceRssi1m": -42,
      "attenuationExponent": 2.4
    }
  ],
  "nodes": [
    { "id": "N1", "name": "Entrada Principal", "x": 0.0, "y": 0.0, "floor": 0 },
    { "id": "N2", "name": "Corredor Central", "x": 0.0, "y": 10.0, "floor": 0 }
  ],
  "edges": [
    {
      "originNodeId": "N1",
      "destinationNodeId": "N2",
      "distanceMeters": 10.0,
      "accessibleVisualImpairment": true,
      "hasTactilePaving": true
    }
  ],
  "pointsOfInterest": [
    {
      "id": "POI_1",
      "anchoredNodeId": "N1",
      "name": "Recepção",
      "type": "INFO_DESK",
      "accessibleDescription": "Balcão de atendimento logo após a porta de vidro."
    }
  ]
}

```

### **Atividade prática:**
<font size=3>
    
1. **Desenho Rápido no Papel:** Desenhe o croqui de um corredor da sua faculdade contendo:
    * 3 salas (ex: "Laboratório 1", "Laboratório 2", "Banheiro").
    * 2 Roteadores Wi-Fi instalados nas paredes.
    * Defina o ponto $(0,0)$ e meça/estime as distâncias em metros.
    <br>

2. **Criação do Arquivo JSON:** Escreva um arquivo chamado `mapa_teste.json` baseado no desenho.

3. **Código de Validação em Kotlin.**

<details>
<summary>Resposta:</summary>

<font size=2>
<br>
    
```kotlin
import com.google.gson.Gson
import java.io.File

// 1. Carregar e ler o arquivo JSON do mapa
val jsonString = File("mapa_teste.json").readText()
val map = Gson().fromJson(jsonString, BuildingMap::class.java)

// 2. Processar e imprimir os dados do mapa
println("=== MAPA CARREGADO: ${map.buildingName} ===")
println("Total de Roteadores cadastrados: ${map.accessPoints.size}")
println("Total de Nós (Interseções): ${map.nodes.size}")
println("Total de POIs: ${map.pointsOfInterest.size}\n")

// 3. Imprimir distância média dos caminhos
val totalDistance = map.edges.sumOf { it.distanceMeters }
println("Distância total de corredores mapeados: ${totalDistance} metros")
```
<br>

</details>
